In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import HTML, display
from transformers import AutoTokenizer

sys.path.insert(0, os.path.abspath("./../../"))

from mllm_shap.connectors.base.audio import SpectrogramGuidedAligner
from mllm_shap.utils.audio import display_audio
from mllm_shap.utils.jupyter import audio_html

/Users/pawelp/Desktop/education/pw/bachelor/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
aligner = SpectrogramGuidedAligner(device=torch.device("cpu"))
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")  # nosec B615

In [ ]:
def process_sample(text_: str, audio_: np.ndarray) -> None:
    """Process a single sample: align audio to text and display results."""
    tokens = [tokenizer.decode([id]) for id in tokenizer.encode(text_)]

    print(f"Aligning to transcript: '{text_}'")
    display(display_audio(audio_))

    print(f"Aligning to {len(tokens)} tokens: {tokens}")
    segments = aligner(audio_, tokens)

    df = pd.DataFrame(
        {
            "Token": [s.token for s in segments],
            "Start Time (s)": [s.start_time for s in segments],
            "End Time (s)": [s.end_time for s in segments],
            "Duration (s)": [s.end_time - s.start_time for s in segments],
            "Confidence": [s.confidence for s in segments],
            "Audio": [s.audio if s.audio else None for s in segments],
        }
    )
    df["Audio"] = df["Audio"].apply(audio_html).str.replace("\n", "")
    display(HTML(df.to_html(escape=False)))


def process_example(ds_: pd.DataFrame) -> None:
    """Process a random example from the dataset."""
    sample = ds_.sample(1)
    text = sample.sentences.values[0]
    audio_male = sample.audio__male.values[0]
    process_sample(text[0], audio_male[0])

In [ ]:
ds = load_dataset(  # nosec B615
    "Pawlo77/mllm-shap",
    name="multi_sentence",
    split="test",
    revision="main",
).to_pandas()

In [5]:
process_example(ds_=ds)

Aligning to transcript: 'Imagine you are a time traveler from the year three thousand.'


Aligning to 15 tokens: ['[CLS]', 'Imagine', 'you', 'are', 'a', 'time', 'travel', '##er', 'from', 'the', 'year', 'three', 'thousand', '.', '[SEP]']


In [6]:
process_example(ds_=ds)

Aligning to transcript: 'Write a startup pitch for a time capsule service.'


Aligning to 15 tokens: ['[CLS]', 'W', '##rite', 'a', 'startu', '##p', 'pitch', 'for', 'a', 'time', 'caps', '##ule', 'service', '.', '[SEP]']


In [7]:
process_example(ds_=ds)

Aligning to transcript: 'Mike's mother had four children.'


Aligning to 10 tokens: ['[CLS]', 'Mike', "'", 's', 'mother', 'had', 'four', 'children', '.', '[SEP]']


In [ ]:
ds = load_dataset(  # nosec B615
    "Pawlo77/mllm-shap",
    name="multi_lingual",
    split="test",
    revision="main",
).to_pandas()

In [9]:
process_example(ds_=ds)

Aligning to transcript: 'Veuillez traduire le passage suivant en espagnol:
Te extraño.'


Aligning to 17 tokens: ['[CLS]', 'Ve', '##uille', '##z', 'trad', '##uire', 'le', 'passage', 'suivant', 'en', 'espagnol', ':', 'Te', 'extra', '##ño', '.', '[SEP]']


In [10]:
process_example(ds_=ds)

Aligning to transcript: 'What is the Japanese title of the novel 【The Catcher in the Rye】?'


Aligning to 20 tokens: ['[CLS]', 'What', 'is', 'the', 'Japanese', 'title', 'of', 'the', 'novel', '【', 'The', 'Catch', '##er', 'in', 'the', 'R', '##ye', '】', '?', '[SEP]']


In [11]:
process_example(ds_=ds)

Aligning to transcript: 'Veuillez inscrire [Préparé] et faire une pause pour des instructions supplémentaires'


Aligning to 22 tokens: ['[CLS]', 'Ve', '##uille', '##z', 'ins', '##crire', '[', 'Pr', '##ép', '##ar', '##é', ']', 'et', 'faire', 'une', 'paus', '##e', 'pour', 'des', 'instructions', 'supplémentaires', '[SEP]']


Conclusion: audio segments are quite well aligned with tokens, however this method should rely on tokenizer that do not include any system tokens and works well with given language. Short tokens seems to be aligned worse than longer ones. Word-level alignment could be a better approach in this case.